<h3>Extracting Features</h3>
Using ConvNeXT Large model to get the features from the image snapshots.

In [5]:
import tensorflow as tf
from tensorflow.keras.applications.convnext import ConvNeXtTiny
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.convnext import preprocess_input
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
import matplotlib.pyplot as plt

print("GPU Available: ", tf.config.list_physical_devices('GPU'))

# Initialize ConvNeXt Small model (without top classification layer)
model = ConvNeXtTiny(weights='imagenet', include_top=False)
model.compile()

def load_and_preprocess_image(img_path, downsampled_res):
    """Load and preprocess image for ConvNeXt"""
    img = image.load_img(img_path, target_size=downsampled_res, color_mode='grayscale')
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    return preprocess_input(img_array)

def extract_features(img_path, downsampled_res):
    """Extract features from image using ConvNeXt Small"""
    preprocessed_img = load_and_preprocess_image(img_path, downsampled_res)
    features = model.predict(preprocessed_img)
    return features.flatten()

def initial_test_run():
    # Example usage with two images
    # Replace these paths with your actual image paths
    snapshot = "data/train_data/train_images/0013.JPG"
    patch_test = "data/train_data/train_images/0014.JPG"

    # Extract features from both images
    print("Extracting features from snapshot...")
    features1 = extract_features(snapshot)

    print("Extracting features from image 2...")
    features2 = extract_features(patch_test)

    # Calculate similarity between features
    similarity = cosine_similarity([features1], [features2])[0][0]

    print(f"Feature vector size: {len(features1)}")
    print(f"Cosine similarity between images: {similarity:.4f}")

    # Higher similarity (closer to 1) suggests images might be from similar locations
    if similarity > 0.8:
        print("High similarity - likely same or nearby location")
    elif similarity > 0.6:
        print("Medium similarity - possibly related location")
    else:
        print("Low similarity - likely different locations")

    plt.figure(figsize=(10, 4))
    plt.hist(features2, bins=50, alpha=0.7, color='blue')
    plt.title("Distribution of patch_test features")
    plt.xlabel("Feature Value")
    plt.ylabel("Frequency")
    plt.show()

# initial_test_run()

GPU Available:  []


In [6]:
def rank_patches_by_distances(snapshot_imgs, patch_imgs, downsampleed_res):
    """Rank patches based on similarity to snapshot features"""
    dist_per_img = []
    
    for snapshot_img in snapshot_imgs:
        dist_patches = []
        print("Matching snapshot image:", snapshot_img)
        snapshot_features = extract_features(snapshot_img, downsampleed_res)
        for patch_img in patch_imgs:
            patch_features = extract_features(patch_img, downsampleed_res)
            dist = euclidean_distances([snapshot_features], [patch_features])[0][0]
            dist_patches.append(dist)
        dist_per_img.append(sorted(enumerate(dist_patches), key=lambda x: x[1], reverse=False))
    
    return dist_per_img


In [7]:
from pathlib import Path

def load_images(sample_size=5):
    """Load a sample of images from the specified folder"""
    # Define the folder path using pathlib
    folder = Path("data/train_data/train_images")

    # List image files with common extensions; note we don't sort them
    image_files = [f for f in folder.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]

    # Get a sample of file paths from the list
    sample = image_files[:sample_size]
    
    return sample

def load_patches():
    """Load patch images from the specified folder"""
    # Define the folder path using pathlib
    folder = Path("map-processing/patches")

    # List image files with common extensions; note we don't sort them
    image_files = [f for f in folder.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    
    return image_files

snapshot_imgs = load_images(5)
patch_imgs = load_patches()
downsampled_res = (100, 133)  # ConvNeXt input size
distances = rank_patches_by_distances(snapshot_imgs, patch_imgs, downsampled_res)

with open("rankings_output.txt", "w") as out_file:
    out_file.write("Patch Rankings by Distance from Snapshots:\n\n")

for i, snapshot in enumerate(snapshot_imgs):
    with open("rankings_output.txt", "a") as out_file:
        out_file.write(f"Snapshot: {snapshot.name}\n")
    for rank, (patch_index, dist) in enumerate(distances[i]):
        patch_name = patch_imgs[patch_index].name
        with open("rankings_output.txt", "a") as out_file:
            patch_name = patch_imgs[patch_index].name
            out_file.write(f"  Rank {rank + 1}: Patch {patch_name} - Distance: {dist:.4f}\n")


Matching snapshot image: data\train_data\train_images\0013.JPG
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0